# Threat Model: OWASP LLM Top 10 (2025)

**Phase 08** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-08/01-owasp-llm-top-10.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 08-01: Threat Model - OWASP LLM Top 10 (2025)

Interactive threat-modeling script. Walk through each OWASP LLM risk,
rate likelihood and impact, and produce a prioritized risk register.

Run with: python main.py
Run headless: python main.py --app-desc "My app description" --auto
"""

import json
import argparse
from dataclasses import dataclass, asdict
from typing import Optional

### OWASP LLM Top 10 (2025) definitions

In [ ]:
OWASP_LLM_TOP_10 = [
    {
        "id": "LLM01",
        "name": "Prompt Injection",
        "description": (
            "User input or retrieved content overrides model instructions. "
            "The model treats injected text as trusted commands."
        ),
        "attack_surface": "User turn, retrieved documents, tool outputs, image alt text",
        "unique_to_llm": True,
    },
    {
        "id": "LLM02",
        "name": "Sensitive Information Disclosure",
        "description": (
            "Model reveals PII, API keys, credentials, or other sensitive data "
            "from training data or context window."
        ),
        "attack_surface": "Training data memorization, context window, direct questioning",
        "unique_to_llm": False,
    },
    {
        "id": "LLM03",
        "name": "Supply Chain",
        "description": (
            "Poisoned or backdoored model weights, datasets, or third-party "
            "components introduce vulnerabilities before deployment."
        ),
        "attack_surface": "Model checkpoints, training datasets, pip/npm dependencies, plugins",
        "unique_to_llm": False,
    },
    {
        "id": "LLM04",
        "name": "Data and Model Poisoning",
        "description": (
            "Training or fine-tuning data is manipulated to alter model behavior, "
            "introduce backdoors, or degrade performance on specific inputs."
        ),
        "attack_surface": "Training pipelines, fine-tuning datasets, RAG index",
        "unique_to_llm": False,
    },
    {
        "id": "LLM05",
        "name": "Improper Output Handling",
        "description": (
            "Model output is passed downstream (rendered as HTML, executed as code, "
            "used in shell commands) without sanitization."
        ),
        "attack_surface": "Frontend rendering, code execution, shell integration, SQL generation",
        "unique_to_llm": False,
    },
    {
        "id": "LLM06",
        "name": "Excessive Agency",
        "description": (
            "The model is granted more permissions, tools, or autonomy than the task requires. "
            "A successful injection can take real-world actions."
        ),
        "attack_surface": "Tool use, agent loops, file system access, API calls, email/calendar",
        "unique_to_llm": True,
    },
    {
        "id": "LLM07",
        "name": "System Prompt Leakage",
        "description": (
            "The contents of the system prompt are extracted by the user via direct "
            "questioning or indirect probing over multiple turns."
        ),
        "attack_surface": "Multi-turn conversation, indirect extraction, context confusion",
        "unique_to_llm": False,
    },
    {
        "id": "LLM08",
        "name": "Vector and Embedding Weaknesses",
        "description": (
            "Poisoned embeddings, adversarial retrieval, or index manipulation cause "
            "the retrieval layer to surface malicious or incorrect content."
        ),
        "attack_surface": "Vector index, embedding pipeline, retrieval ranking",
        "unique_to_llm": False,
    },
    {
        "id": "LLM09",
        "name": "Misinformation",
        "description": (
            "Model generates plausible but false information (hallucination) "
            "that users trust and act on."
        ),
        "attack_surface": "Any generation task without ground-truth grounding or citation",
        "unique_to_llm": False,
    },
    {
        "id": "LLM10",
        "name": "Unbounded Consumption",
        "description": (
            "No limits on token usage, API calls, or cost enable denial-of-service "
            "attacks or runaway spend."
        ),
        "attack_surface": "Public endpoints, agent loops, streaming responses",
        "unique_to_llm": False,
    },
]

### Risk register data structures

In [ ]:
@dataclass
class RiskEntry:
    id: str
    name: str
    description: str
    attack_surface: str
    unique_to_llm: bool
    likelihood: int  # 1=rare, 2=possible, 3=likely
    impact: int      # 1=low, 2=medium, 3=high
    notes: str
    score: int       # likelihood * impact (1-9)
    priority: str    # CRITICAL (7-9), HIGH (5-6), MEDIUM (3-4), LOW (1-2)

    @staticmethod
    def priority_label(score: int) -> str:
        if score >= 7:
            return "CRITICAL"
        elif score >= 5:
            return "HIGH"
        elif score >= 3:
            return "MEDIUM"
        return "LOW"

### Interactive threat-modeling session

In [ ]:
def rate_risk_interactive(risk: dict) -> RiskEntry:
    """Prompt the engineer to rate likelihood and impact for a single risk."""
    print(f"\n{'='*60}")
    print(f"{risk['id']}: {risk['name']}")
    if risk["unique_to_llm"]:
        print("  [UNIQUE TO LLM SYSTEMS - no traditional web security analogue]")
    print(f"  Description: {risk['description']}")
    print(f"  Attack surface: {risk['attack_surface']}")
    print()

    while True:
        try:
            likelihood = int(input("  Rate LIKELIHOOD (1=rare, 2=possible, 3=likely): ").strip())
            if likelihood in (1, 2, 3):
                break
            print("  Please enter 1, 2, or 3.")
        except ValueError:
            print("  Please enter 1, 2, or 3.")

    while True:
        try:
            impact = int(input("  Rate IMPACT (1=low, 2=medium, 3=high): ").strip())
            if impact in (1, 2, 3):
                break
            print("  Please enter 1, 2, or 3.")
        except ValueError:
            print("  Please enter 1, 2, or 3.")

    notes = input("  Notes (optional, press Enter to skip): ").strip()

    score = likelihood * impact
    priority = RiskEntry.priority_label(score)
    print(f"  => Score: {score}/9  Priority: {priority}")

    return RiskEntry(
        id=risk["id"],
        name=risk["name"],
        description=risk["description"],
        attack_surface=risk["attack_surface"],
        unique_to_llm=risk["unique_to_llm"],
        likelihood=likelihood,
        impact=impact,
        notes=notes,
        score=score,
        priority=priority,
    )


def auto_rate_risk(risk: dict, likelihood: int = 2, impact: int = 2) -> RiskEntry:
    """Rate a risk automatically (for testing/demo mode)."""
    # Assign defaults; LLM01 and LLM06 default to 3x3 = CRITICAL
    if risk["id"] in ("LLM01", "LLM06"):
        likelihood, impact = 3, 3
    elif risk["id"] in ("LLM02", "LLM07"):
        likelihood, impact = 3, 2
    elif risk["id"] in ("LLM09", "LLM05"):
        likelihood, impact = 2, 2
    else:
        likelihood, impact = 1, 2

    score = likelihood * impact
    return RiskEntry(
        id=risk["id"],
        name=risk["name"],
        description=risk["description"],
        attack_surface=risk["attack_surface"],
        unique_to_llm=risk["unique_to_llm"],
        likelihood=likelihood,
        impact=impact,
        notes="Auto-rated for demo",
        score=score,
        priority=RiskEntry.priority_label(score),
    )

### Risk register output

In [ ]:
def print_risk_register(entries: list[RiskEntry], app_desc: str) -> None:
    """Print the sorted risk register."""
    sorted_entries = sorted(entries, key=lambda r: r.score, reverse=True)

    print("\n" + "=" * 70)
    print("RISK REGISTER")
    print("=" * 70)
    print(f"Application: {app_desc[:65]}")
    print()
    print(f"{'Rank':<5} {'ID':<7} {'Name':<32} {'L':>2} {'I':>2} {'Score':>5}  {'Priority'}")
    print("-" * 70)

    for rank, entry in enumerate(sorted_entries, 1):
        llm_marker = "*" if entry.unique_to_llm else " "
        print(
            f"{rank:<5} {entry.id:<7} {entry.name[:30]:<32} "
            f"{entry.likelihood:>2} {entry.impact:>2} {entry.score:>5}  {entry.priority}{llm_marker}"
        )

    print()
    print("* = Risk unique to LLM systems (no traditional web security analogue)")
    print()

    critical = [e for e in sorted_entries if e.priority == "CRITICAL"]
    high = [e for e in sorted_entries if e.priority == "HIGH"]

    if critical:
        print(f"CRITICAL risks ({len(critical)}) - address before launch:")
        for e in critical:
            print(f"  {e.id}: {e.name}")
            if e.notes and e.notes != "Auto-rated for demo":
                print(f"    Notes: {e.notes}")

    if high:
        print(f"\nHIGH risks ({len(high)}) - address in first sprint:")
        for e in high:
            print(f"  {e.id}: {e.name}")
            if e.notes and e.notes != "Auto-rated for demo":
                print(f"    Notes: {e.notes}")


def save_risk_register(entries: list[RiskEntry], app_desc: str, path: str = "risk_register.json") -> None:
    """Save the risk register to JSON."""
    sorted_entries = sorted(entries, key=lambda r: r.score, reverse=True)
    output = {
        "application": app_desc,
        "owasp_version": "LLM Top 10 2025",
        "risks": [asdict(e) for e in sorted_entries],
    }
    with open(path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nRisk register saved to {path}")

### Main

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Lesson 08-01: OWASP LLM Top 10 Threat Model")
    parser.add_argument(
        "--app-desc",
        default="",
        help="One-sentence description of your LLM application",
    )
    parser.add_argument(
        "--auto",
        action="store_true",
        help="Use default ratings (demo mode, no interactive input)",
    )
    parser.add_argument(
        "--output",
        default="risk_register.json",
        help="Output path for the risk register JSON",
    )
    args = parser.parse_args()

    print("\n=== OWASP LLM Top 10 (2025) Threat Modeling Session ===\n")

    app_desc = args.app_desc
    if not app_desc and not args.auto:
        app_desc = input("Describe your LLM application in one sentence:\n> ").strip()
    elif not app_desc:
        app_desc = "Demo: RAG-based customer support assistant (no auth, public-facing)"

    print(f"\nApplication: {app_desc}")
    print(f"\nYou will rate each of the 10 OWASP LLM risks on:")
    print("  Likelihood: 1=rare, 2=possible, 3=likely")
    print("  Impact:     1=low,  2=medium,   3=high")
    print("  Score = Likelihood x Impact (max 9)")
    print("  Priority: CRITICAL (7-9), HIGH (5-6), MEDIUM (3-4), LOW (1-2)")

    entries: list[RiskEntry] = []

    for risk in OWASP_LLM_TOP_10:
        if args.auto:
            entry = auto_rate_risk(risk)
        else:
            entry = rate_risk_interactive(risk)
        entries.append(entry)

    print_risk_register(entries, app_desc)
    save_risk_register(entries, app_desc, args.output)

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Which OWASP LLM risk does this represent, and why is it categorized as unique to LLM systems?**

- A. LLM07 (System Prompt Leakage), because the model revealed its instructions; this is unique to LLMs because traditional applications do not expose their business logic through user input
- B. LLM01 (Prompt Injection), because user input modified the model's behavior; this is unique to LLMs because no previous application class allowed English sentences to change application logic
- C. LLM02 (Sensitive Information Disclosure), because the system prompt is sensitive data; this is analogous to an information disclosure bug in any web application
- D. LLM05 (Improper Output Handling), because the model output contained system configuration; this is the same as a verbose error message in a traditional web app

**2. Which two OWASP LLM risks should be rated CRITICAL for this architecture and why?**

- A. LLM09 (Misinformation) and LLM10 (Unbounded Consumption), because summarization can produce hallucinations and Slack posting could be called repeatedly
- B. LLM01 (Prompt Injection) and LLM06 (Excessive Agency), because malicious PDFs can inject instructions and the Slack posting capability means a successful injection can act on the attacker's behalf
- C. LLM03 (Supply Chain) and LLM04 (Data Poisoning), because the shared Drive folder could be used to poison the model's training data
- D. LLM07 (System Prompt Leakage) and LLM08 (Vector Weaknesses), because the agent reads documents and has an embedding-based retrieval pipeline

**3. What is the correct reasoning for keeping LLM03 at LOW for a RAG service that uses a hosted Claude API with no fine-tuning and no third-party model weights?**

- A. Supply chain risk only applies to open-source models; SaaS APIs have no supply chain
- B. The primary supply chain attack surfaces (model weights, training data) are controlled by Anthropic, not you; your attack surface is limited to your pip dependencies, which are a standard software risk you already manage
- C. LLM03 is theoretical and has not been exploited in practice, so it should always be rated LOW
- D. Supply chain risk scales with the number of dependencies; a simple RAG service has few dependencies

**4. Why is adding a confirmation step for delete operations an incomplete mitigation for LLM06?**

- A. Confirmation steps slow down the user experience and are not a valid security control
- B. The confirmation step only addresses one tool; the model still has unconstrained access to email, calendar, and read operations, any of which could be abused via prompt injection
- C. The real fix is to remove all tools and switch to a retrieval-only architecture
- D. Confirmation steps can themselves be bypassed via prompt injection by having the model answer its own confirmation prompt

**5. What is the correct critique of this risk prioritization?**

- A. The prioritization is correct; cost control is a business-critical concern and the disclaimer handles misinformation liability
- B. LLM09 should be CRITICAL in a medical domain because users act on health information regardless of disclaimers, while LLM10 can be mitigated with standard rate limiting; the business impact of a medical misinformation incident is higher than cost overrun
- C. Both should be CRITICAL; medical chatbots have the highest possible risk across all 10 categories
- D. LLM09 should be LOW because the model is citing medical literature; hallucination rates for factual recall are low

**6. Under what condition would LLM08 become a relevant risk for this architecture?**

- A. Never; LLM08 only applies to public-facing systems where external attackers can reach the vector index
- B. If the HR handbook update process allows a compromised HR account or a malicious insider to insert documents that cause the retrieval layer to return misleading policy information
- C. If the embedding model provider changes their model, causing retrieval quality to degrade
- D. LLM08 is always LOW because vector databases have no known attack vectors

**7. Which statement correctly describes how the risk registers should differ?**

- A. The registers should be identical because the OWASP risks are fixed categories that do not depend on architecture
- B. App B should have higher scores on LLM01, LLM06, and LLM10 because public access increases injection surface, email sending creates agency risk, and no auth enables DoS; App A should score lower on most risks due to limited attack surface
- C. App A should score higher because internal users have higher trust and are less likely to inject malicious prompts
- D. The only difference is LLM10; public apps have more consumption risk but all other risks are the same

**8. What does this incident reveal about your original threat model reasoning, and what should change?**

- A. The threat model was correct; if the system prompt has no secrets, LLM07 has no impact and LOW is the right rating
- B. The reasoning was flawed: even a system prompt without explicit secrets reveals your filtering logic, jailbreak resistance, and behavioral constraints, which attackers use to craft targeted bypasses; LLM07 should be rated based on what exposure enables, not just what is literally in the prompt
- C. The incident is a false positive; users already know the model has a system prompt, so leaking it is not a security issue
- D. The fix is to put secrets in environment variables, after which LLM07 drops to LOW with no further action needed

_Answers are in `checks.json` in the lesson directory._